## 4. LSTM Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import LSTMPredictor, HORIZONS

MODEL_NAME = 'LSTM'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'lstm'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'lstm_stability_summary.json'

In [3]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [4]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)  # before constructing the model -- weight init draws from this
    model = LSTMPredictor()

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'lstm' /f'lstm_seed{seed}',
        verbose=False,  # quiet per-epoch text logging -- the progress bar below (and the one
                         # summary line per seed, after) carry that job instead
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    # Same test predictions, broken down per participant instead of pooled -- this is what lets
    # the aggregation cells separate "variability across seeds" from "variability across
    # participants" instead of only ever reporting the former.
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'lstm_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    print(f"\n{MODEL_NAME} | seed={seed} | stopped at epoch {n_epochs}/50")

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )




Test results — LSTM | seed=7

device: cpu  model: LSTMPredictor  params: 207,749


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]


LSTM | seed=7 | stopped at epoch 28/50
 15 min | RMSE 11.39 | MAE 7.75 | gRMSE 12.85 | Zone A 97.05% | A+B 99.77%
 30 min | RMSE 18.59 | MAE 12.36 | gRMSE 21.70 | Zone A 90.38% | A+B 99.52%
 60 min | RMSE 27.34 | MAE 18.08 | gRMSE 33.46 | Zone A 81.32% | A+B 98.92%
 90 min | RMSE 32.51 | MAE 21.73 | gRMSE 40.52 | Zone A 75.45% | A+B 98.46%
120 min | RMSE 35.50 | MAE 24.01 | gRMSE 44.60 | Zone A 71.58% | A+B 98.11%

Test results — LSTM | seed=14

device: cpu  model: LSTMPredictor  params: 207,749


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]


LSTM | seed=14 | stopped at epoch 35/50
 15 min | RMSE 11.25 | MAE 7.62 | gRMSE 12.50 | Zone A 97.13% | A+B 99.84%
 30 min | RMSE 18.51 | MAE 12.26 | gRMSE 21.52 | Zone A 90.57% | A+B 99.57%
 60 min | RMSE 27.22 | MAE 18.03 | gRMSE 33.22 | Zone A 81.76% | A+B 99.01%
 90 min | RMSE 32.49 | MAE 21.78 | gRMSE 40.38 | Zone A 75.38% | A+B 98.36%
120 min | RMSE 35.56 | MAE 24.09 | gRMSE 44.59 | Zone A 71.17% | A+B 98.13%

Test results — LSTM | seed=21

device: cpu  model: LSTMPredictor  params: 207,749


...


LSTM | seed=56 | stopped at epoch 45/50
 15 min | RMSE 11.25 | MAE 7.56 | gRMSE 12.42 | Zone A 97.10% | A+B 99.88%
 30 min | RMSE 18.55 | MAE 12.28 | gRMSE 21.43 | Zone A 90.49% | A+B 99.60%
 60 min | RMSE 27.38 | MAE 18.17 | gRMSE 33.34 | Zone A 81.70% | A+B 99.05%
 90 min | RMSE 32.68 | MAE 21.91 | gRMSE 40.63 | Zone A 75.31% | A+B 98.43%
120 min | RMSE 35.71 | MAE 24.25 | gRMSE 44.81 | Zone A 70.99% | A+B 98.06%

Test results — LSTM | seed=63

device: cpu  model: LSTMPredictor  params: 207,749


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]


LSTM | seed=63 | stopped at epoch 27/50
 15 min | RMSE 11.39 | MAE 7.71 | gRMSE 12.67 | Zone A 97.06% | A+B 99.87%
 30 min | RMSE 18.69 | MAE 12.35 | gRMSE 21.76 | Zone A 90.51% | A+B 99.58%
 60 min | RMSE 27.54 | MAE 18.00 | gRMSE 33.77 | Zone A 81.68% | A+B 98.91%
 90 min | RMSE 32.91 | MAE 21.70 | gRMSE 41.18 | Zone A 75.41% | A+B 98.23%
120 min | RMSE 36.12 | MAE 24.06 | gRMSE 45.61 | Zone A 71.81% | A+B 97.75%

Test results — LSTM | seed=70

device: cpu  model: LSTMPredictor  params: 207,749


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]


LSTM | seed=70 | stopped at epoch 35/50
 15 min | RMSE 11.25 | MAE 7.65 | gRMSE 12.36 | Zone A 96.99% | A+B 99.81%
 30 min | RMSE 18.39 | MAE 12.19 | gRMSE 21.17 | Zone A 90.45% | A+B 99.56%
 60 min | RMSE 27.00 | MAE 17.91 | gRMSE 32.77 | Zone A 81.48% | A+B 98.94%
 90 min | RMSE 32.23 | MAE 21.64 | gRMSE 40.02 | Zone A 75.53% | A+B 98.38%
120 min | RMSE 35.26 | MAE 23.85 | gRMSE 44.17 | Zone A 71.73% | A+B 98.06%


In [ ]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [
                r['test_results'][h]['ceg']['A']
            for r in all_results
            ]
        elif m == 'zone_ab':
            seed_values = [
                r['test_results'][h]['ceg']['A']
                + r['test_results'][h]['ceg']['B']
                for r in all_results
            ]
        else:
            seed_values = [
                r['test_results'][h][m]
                for r in all_results
            ]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [
                _extract(r['test_results_per_participant'][pid][h], m)
                for r in all_results
            ]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }

# ---------------------------------------------------------
# Print: mean +/- sd1 (sd2)
# ---------------------------------------------------------
print(
    f'=== {MODEL_NAME} test set: '
    f'overall mean +/- SD_seed (SD_participant) ==='
)

for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- "
        f"{s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- "
        f"{s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- "
        f"{s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- "
        f"{s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
        f"{s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

In [ ]:
# Save one final summary JSON
with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')